In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 130)


In [3]:
features = [

    "patent_count",

    "trademark_count",

    "technology_domain",

    "api_available",

    "mobile_app_available",

    "web_platform_available"

]

for col in features:
    print(col, "=>", col in df.columns)

patent_count => True
trademark_count => True
technology_domain => True
api_available => True
mobile_app_available => True
web_platform_available => True


In [4]:
tech_encoder = {
    
    "AI":100,
    "Blockchain":90,
    "IoT":85,
    "Cybersecurity":85,
    "Cloud":80,
    "Data Science":80,
    "FinTech":75,
    "HealthTech":75,
    "SaaS":70

}

df["technology_score"] = (

    df["technology_domain"]

    .map(tech_encoder)

    .fillna(50)

)

In [12]:
df["patent_score"] = np.clip(
    (df["patent_count"] / df["patent_count"].max()) * 100,
    0,
    100
)

In [13]:
df["trademark_score"] = np.clip(
    (df["trademark_count"] / df["trademark_count"].max()) * 100,
    0,
    100
)

In [14]:
df["platform_score"] = (

    (df["api_available"] > 0).astype(int) * 40 +

    (df["mobile_app_available"] > 0).astype(int) * 30 +

    (df["web_platform_available"] > 0).astype(int) * 30

)

In [15]:
df["innovation_score"] = (

      0.30 * df["technology_score"]

    + 0.30 * df["patent_score"]

    + 0.15 * df["trademark_score"]

    + 0.25 * df["platform_score"]

)

df["innovation_score"] = np.clip(

    df["innovation_score"],

    0,

    100

)

In [16]:
print(

    df["innovation_score"]

    .describe()

)

count    50000.000000
mean        62.331053
std          9.935718
min         15.075000
25%         54.895000
50%         62.425000
75%         69.910000
max         84.850000
Name: innovation_score, dtype: float64


In [17]:
df.to_csv(

    "../datasets/cleaned/startup_info.csv",

    index=False

)

print("Innovation Score Added ✅")

Innovation Score Added ✅


In [18]:
print(df["innovation_score"].describe())

count    50000.000000
mean        62.331053
std          9.935718
min         15.075000
25%         54.895000
50%         62.425000
75%         69.910000
max         84.850000
Name: innovation_score, dtype: float64


In [19]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

In [20]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 135)


In [21]:
def innovation_label(score):

    if score < 45:
        return "Low"

    elif score < 70:
        return "Medium"

    else:
        return "High"


df["innovation_label"] = (
    df["innovation_score"]
    .apply(innovation_label)
)

print(
    df["innovation_label"]
    .value_counts()
)

innovation_label
Medium    35983
High      12381
Low        1636
Name: count, dtype: int64


In [22]:
features = [

    "patent_count",

    "trademark_count",

    "api_available",

    "mobile_app_available",

    "web_platform_available",

    "technology_score"

]

X = df[features]

y = df["innovation_label"]

In [23]:
print(X.dtypes)

print(
    X.isnull()
    .sum()
)

patent_count                int64
trademark_count             int64
api_available               int64
mobile_app_available        int64
web_platform_available      int64
technology_score          float64
dtype: object
patent_count              0
trademark_count           0
api_available             0
mobile_app_available      0
web_platform_available    0
technology_score          0
dtype: int64


In [24]:
le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [25]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [26]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [27]:
preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 99.55 %
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2476
           1       0.99      0.96      0.97       327
           2       0.99      1.00      1.00      7197

    accuracy                           1.00     10000
   macro avg       0.99      0.98      0.99     10000
weighted avg       1.00      1.00      1.00     10000



In [28]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                  Feature  Importance
0            patent_count    0.690667
1         trademark_count    0.266477
3    mobile_app_available    0.015496
2           api_available    0.013868
4  web_platform_available    0.012585
5        technology_score    0.000907


In [30]:
joblib.dump(

    model,

    "../models/innovation_assessment_model/innovation_assessment_model.pkl"

)

joblib.dump(

    le,

    "../models/innovation_assessment_model/label_encoder.pkl"

)

print("Innovation Assessment Model Saved ✅")

Innovation Assessment Model Saved ✅


In [31]:
metadata = {

    "model_name":
        "Innovation Assessment Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/innovation_assessment_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [32]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Saved ✅")

Dataset Saved ✅


In [33]:
print(df["innovation_label"].value_counts())

print("Accuracy:", acc)

print(importance)

innovation_label
Medium    35983
High      12381
Low        1636
Name: count, dtype: int64
Accuracy: 0.9955
                  Feature  Importance
0            patent_count    0.690667
1         trademark_count    0.266477
3    mobile_app_available    0.015496
2           api_available    0.013868
4  web_platform_available    0.012585
5        technology_score    0.000907


In [37]:
print(df.columns.tolist())


['startup_name', 'city', 'state', 'country', 'sector', 'sub_sector', 'founded_year', 'startup_stage', 'funding_stage', 'total_funding', 'investor_count', 'founder_names', 'founder_count', 'patent_count', 'technology_domain', 'employee_count', 'market_growth_rate', 'market_size', 'current_status', 'source_url', 'data_source', 'website', 'linkedin_url', 'twitter_url', 'github_url', 'crunchbase_url', 'logo_url', 'latitude', 'longitude', 'industry_keywords', 'business_model', 'target_market', 'b2b_b2c', 'market_type', 'pricing_model', 'revenue_model', 'industry_growth_rate', 'industry_risk_score', 'company_type', 'registration_type', 'dpiit_registered', 'gst_registered', 'international_presence', 'number_of_offices', 'female_founder_count', 'max_founder_experience', 'iit_founder', 'iim_founder', 'foreign_university_founder', 'trademark_count', 'mobile_app_available', 'web_platform_available', 'api_available', 'tam', 'sam', 'som', 'customer_segment_count', 'international_market_access', 'la